# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

I already ran the starter pipeline end to end in Notebooks 1 and 2, so I have first-hand, reproducible evidence that this lane has real signal to work with: a transparent hand-written baseline reaches Precision@50 = 0.240 on this data, while a random forest trained on the same observable features reaches 0.740 — about 3x more of the top 50 flagged pages are genuinely correct (numbers reproduced in Section 3 below). That gap is direct evidence the underlying pattern is real but too tangled for a single hand rule, which is exactly the condition under which ML earns its place instead of a plain if-statement. The lane also produces the most directly actionable output of the four predefined lanes — a ranked review queue with reason codes a human can inspect, not just a report — and it maps cleanly onto a real FlyRank workflow: a content team with limited review capacity needs to know which pages to look at first. I'll confirm or revisit this choice by the end of Week 4 once I've seen the full warehouse release, but starting from what I've already verified in the starter data is the strongest footing I have right now.

In [1]:
import os
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # work/notebooks -> repo root

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages across {df['client_id'].nunique()} clients — enough breadth to check if a")
print("refresh-priority signal holds across more than one account, not just a single client's quirks.")

30,000 pages across 32 clients — enough breadth to check if a
refresh-priority signal holds across more than one account, not just a single client's quirks.


## 2. The question: decision, action, cost of a wrong call

**Decision this improves:** out of everything in a client's content library, which pages should a content reviewer look at *first* this cycle, given they can only realistically review a limited number (e.g. the top 50)?

**Who acts, and what they do:** a content strategist/editor works down the ranked queue and, for each flagged page, picks a concrete action — refresh the content, review its CTR/metadata, review on-page engagement, expand it, or simply keep monitoring it. The reason code attached to each page is what tells them which.

**Cost of a wrong call:**
- *False positive* (a page is flagged urgent but wasn't actually declining): burns a reviewer's limited hours on a page that didn't need it — and because capacity is fixed, a real problem page effectively gets pushed further down the queue and reviewed later than it should be.
- *False negative* (a genuinely declining page isn't surfaced near the top): the page keeps losing visibility/traffic unnoticed, and by the time it's caught the drop is bigger and more expensive to recover from than if it had been flagged early.

Because review capacity — not overall accuracy — is the real constraint, **Precision@50** (or whatever K matches the team's real capacity) is the metric that matches the actual decision: it asks "of the top K we'd actually act on, how many are right," not "how good is the model on average."

**Why data/ML helps, not just a rule:** the evidence is already in this dataset (Section 3): a single hand-written rule tops out around Precision@50 = 0.240, while a random forest trained on the same observable signals reaches 0.740. Whether a page is actually declining depends on several signals moving together — impressions, position, CTR, freshness, engagement, content type — in ways that are real but too tangled to hand-write as one if/else rule. That's exactly the case where a learned model earns its place over a plain rule.

In [2]:
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")
high_vis_declining = eligible[(eligible["impressions_90d"] >= 500) & (eligible["trend_direction"] == "down")]

print(f"{len(high_vis_declining):,} pages are BOTH high-visibility (impressions_90d >= 500) AND currently")
print("declining — real traffic already flowing, already trending down. Missing these on a review")
print("queue is the concrete cost of a false negative; flagging the wrong ones burns a reviewer's")
print("limited hours instead.")

9,961 pages are BOTH high-visibility (impressions_90d >= 500) AND currently
declining — real traffic already flowing, already trending down. Missing these on a review
queue is the concrete cost of a false negative; flagging the wrong ones burns a reviewer's
limited hours instead.


## 3. Quick look at the data (2-3 real numbers)

Two things ground the case for this lane in real numbers: how much a learned ranking already beats a hand rule on this data, and how concentrated the risk is among pages that still carry real traffic.

In [3]:
# Baseline vs. learned model — from the committed, reproducible outputs/model_report.md
# (regenerate anytime with `python scripts/run_all.py`)
with open("outputs/model_report.md") as f:
    lines = f.read().splitlines()

rows = {}
for line in lines:
    if line.startswith("| baseline_rules") or line.startswith("| random_forest"):
        cols = [c.strip() for c in line.strip("|").split("|")]
        rows[cols[0]] = float(cols[3])  # Precision@50 column

baseline_p50, rf_p50 = rows["baseline_rules"], rows["random_forest"]
print(f"Baseline hand-rule  Precision@50: {baseline_p50:.3f}  (~{round(baseline_p50*50)} of the top 50 right)")
print(f"Random forest       Precision@50: {rf_p50:.3f}  (~{round(rf_p50*50)} of the top 50 right)")
print(f"Random forest is {rf_p50/baseline_p50:.2f}x the baseline on this metric.\n")

# Decline rate overall vs. among high-visibility pages — computed live from the starter CSV
decline_rate = (eligible["trend_direction"] == "down").mean()
high_vis = eligible[eligible["impressions_90d"] >= 500]
high_vis_rate = (high_vis["trend_direction"] == "down").mean()

print(f"Overall decline rate: {decline_rate:.1%} "
      f"({int((eligible['trend_direction']=='down').sum()):,} of {len(eligible):,} eligible pages)")
print(f"Decline rate among high-visibility pages only: {high_vis_rate:.1%} "
      f"({(high_vis_rate - decline_rate) * 100:+.1f} percentage points above the overall rate)")

Baseline hand-rule  Precision@50: 0.240  (~12 of the top 50 right)
Random forest       Precision@50: 0.740  (~37 of the top 50 right)
Random forest is 3.08x the baseline on this metric.

Overall decline rate: 54.2% (16,262 of 30,000 eligible pages)
Decline rate among high-visibility pages only: 59.6% (+5.3 percentage points above the overall rate)


## 4. Careful words: what I can and can't claim

**What I can claim:** observed, directional patterns from this dataset — e.g. "pages with this combination of signals were associated with decline in this sample," and decision-support statements — "this ranking would point reviewers at more true declines per hour spent than the hand rule." Every number above comes from client-holdout validation on a single 30,000-row anonymized starter slice, so I'll phrase results as "in this sample" / "on this slice," not as a universal law.

**What I will never claim:** that a model "proves" a Google ranking factor; that a refresh recommendation is guaranteed to cause a recovery (that needs a controlled experiment, not this observational data); or that a wrong call is harmless. I'll also stay honest about the label itself — `is_declining_label` is `trend_direction` computed from the *current* window, a proxy, not a genuinely future-observed outcome. A stronger version of this lane, once I move to the full warehouse release, should use a prior-90-days -> next-30-days label instead. And I won't publish client names, URLs, or raw queries — the starter data is pseudonymized specifically so that risk doesn't exist here, and I'll keep that same discipline once I'm working with the warehouse release.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.